# Activation Functions - Exercises

Ten graded exercises covering values, derivatives, gates, softmax, and diagnostics.

In [ ]:
import numpy as np

def header(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

def check_close(name, value, expected, tol=1e-7):
    ok = np.allclose(value, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}: value={value}, expected={expected}")
    return ok

def check_true(name, condition):
    ok = bool(condition)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    return ok

def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(z):
    shifted = z - np.max(z)
    e = np.exp(shifted)
    return e/e.sum()
print("Exercise helpers ready.")

## Exercise 1: Sigmoid derivative (*)

Derive and compute sigmoid derivative.

In [ ]:
x=np.array([-1.0,0.0,1.0])
deriv=None
print(deriv)

In [ ]:
header("Exercise 1: Sigmoid derivative")
x=np.array([-1.0,0.0,1.0])
s=sigmoid(x)
deriv=s*(1-s)
check_close("center derivative", deriv[1], 0.25)
print("\nTakeaway: sigmoid saturates because its derivative approaches zero in both tails.")

## Exercise 2: Tanh derivative (*)

Compute tanh derivative.

In [ ]:
x=np.array([-1.0,0.0,1.0])
deriv=None
print(deriv)

In [ ]:
header("Exercise 2: Tanh derivative")
x=np.array([-1.0,0.0,1.0])
deriv=1-np.tanh(x)**2
check_close("center derivative", deriv[1], 1.0)
print("\nTakeaway: tanh is zero-centered but still saturates.")

## Exercise 3: ReLU family (*)

Compute ReLU and Leaky ReLU.

In [ ]:
x=np.array([-2.0,0.0,3.0])
y=None
print(y)

In [ ]:
header("Exercise 3: ReLU family")
x=np.array([-2.0,0.0,3.0])
relu=np.maximum(0,x)
leaky=np.where(x>0,x,0.1*x)
check_close("relu", relu, np.array([0.0,0.0,3.0]))
check_true("leaky keeps negative signal", leaky[0]<0)
print("\nTakeaway: Leaky ReLU reduces the dead-neuron zero-gradient region.")

## Exercise 4: Affine collapse (**)

Show two affine layers collapse without activation.

In [ ]:
W1=np.eye(2); W2=2*np.eye(2); x=np.ones(2)
out=None
print(out)

In [ ]:
header("Exercise 4: Affine collapse")
W1=np.array([[1.,2.],[0.,1.]]); W2=np.array([[2.,0.],[1.,1.]])
b1=np.array([1.,-1.]); b2=np.array([0.5,0.5]); x=np.array([2.,3.])
out=W2@(W1@x+b1)+b2
A=W2@W1; c=W2@b1+b2
check_close("collapsed affine", out, A@x+c)
print("\nTakeaway: activations are required for nonlinear depth.")

## Exercise 5: Stable softmax (**)

Implement stable softmax and test shift invariance.

In [ ]:
z=np.array([1000.,999.,998.])
p=None
print(p)

In [ ]:
header("Exercise 5: Stable softmax")
z=np.array([1000.,999.,998.])
p=softmax(z)
check_close("sums to one", p.sum(), 1.0)
check_close("shift invariant", p, softmax(z-1000))
print("\nTakeaway: subtracting the max protects softmax from overflow.")

## Exercise 6: Softmax Jacobian (**)

Compute the softmax Jacobian.

In [ ]:
s=np.array([0.2,0.3,0.5])
J=None
print(J)

In [ ]:
header("Exercise 6: Softmax Jacobian")
s=np.array([0.2,0.3,0.5])
J=np.diag(s)-np.outer(s,s)
check_close("row sums zero", J.sum(axis=1), np.zeros(3))
check_true("PSD", np.linalg.eigvalsh(J).min()>-1e-10)
print("\nTakeaway: softmax derivatives are coupled across classes.")

## Exercise 7: GELU vs SiLU (**)

Compute smooth activations at sample inputs.

In [ ]:
x=np.array([-1.,0.,1.])
gel=None
print(gel)

In [ ]:
header("Exercise 7: GELU vs SiLU")
x=np.array([-1.,0.,1.])
gel=0.5*x*(1+np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3)))
silu=x*sigmoid(x)
check_true("finite smooth activations", np.isfinite(gel).all() and np.isfinite(silu).all())
print("GELU", gel, "SiLU", silu)
print("\nTakeaway: smooth activations keep small negative outputs instead of hard zeroing.")

## Exercise 8: GLU (***)

Compute GLU output and local derivatives.

In [ ]:
a=np.array([2.,-1.]); b=np.array([0.,2.])
out=None
print(out)

In [ ]:
header("Exercise 8: GLU")
a=np.array([2.,-1.]); b=np.array([0.,2.])
g=sigmoid(b); out=a*g
check_close("GLU output", out, a*g)
check_close("dy/da", g, sigmoid(b))
print("\nTakeaway: gated activations create separate content and gate gradient paths.")

## Exercise 9: He variance (***)

Compute Xavier and He variances.

In [ ]:
n_in=128; n_out=64
var=None
print(var)

In [ ]:
header("Exercise 9: He variance")
n_in=128; n_out=64
xavier=2/(n_in+n_out); he=2/n_in
check_true("He greater than Xavier here", he>xavier)
print("xavier", xavier, "he", he)
print("\nTakeaway: initialization depends on activation statistics.")

## Exercise 10: Dead ReLU diagnostic (***)

Detect dead units from preactivation statistics.

In [ ]:
preacts=np.array([[-1.,2.],[-2.,3.]])
dead=None
print(dead)

In [ ]:
header("Exercise 10: Dead ReLU diagnostic")
preacts=np.array([[-2.,1.],[-1.,2.],[-3.,3.]])
active_fraction=np.mean(preacts>0, axis=0)
dead=active_fraction<0.01
check_true("first unit dead", dead[0])
print("\nTakeaway: dead ReLUs are diagnosed by persistently inactive preactivations.")